# 3.3 Evaluate and compare

**What this notebook is for.** Turning a sweep's metric table into a defensible
answer to "which configuration should we ship". It is the notebook that produces
the artefacts a review looks at, so the plots matter as much as the numbers.

**Why the metric is top-k and not AUC.** Fraud review is a queue with a fixed
capacity. Nobody reviews 100,000 alerts; somebody reviews the top 500. AUC
answers "is the ranking good overall", which is not the question. `K_GRID_FIXED`
and `K_GRID_PERCENTAGES` in `common/metrics/ranking.py` define the cut points,
and `make_k_grid` adapts them to the population size, so a metric at k=500 means
the same thing across scenarios of different sizes.

**Why aggregation level matters.** `AGGREGATION_LEVELS` lets the same scores be
evaluated per record, per case, or per customer. A model can look excellent per
record and mediocre per customer, which happens when it flags many rows of a few
customers. Per-customer is usually the honest number, because a reviewer works a
customer, not a row.

**Why `require_all_scenarios` defaults to on.** A configuration that wins on one
scenario and was never scored on the others has not won. `leaderboard` and
`select_best_config` both take the flag; turning it off is how you get a champion
that was never tested on the population it will run against.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

In [ ]:
from trust_score_05.ml.config import load_ml_config

ENV = "dev"
cfg = load_ml_config(env=ENV)

METRIC = "precision_at_k"
LEVEL = "customer"
K_PERCENT = 1.0

print("scenarios          :", cfg.scenarios)
print("aggregation levels :", cfg.aggregation_levels)
print("metric             :", METRIC)
print("level              :", LEVEL)

## Load the sweep summary

In [ ]:
import pandas as pd

from trust_score_05.common.io.s3 import read_json, ready_exists
from trust_score_05.ml.paths import sweep_summary_prefix

prefix = sweep_summary_prefix(cfg)
print("reading from:", prefix)
print("ready       :", ready_exists(prefix))

SUMMARY_PATH = None  # set to a local csv/parquet to work offline

if SUMMARY_PATH:
    metrics = pd.read_parquet(SUMMARY_PATH) if SUMMARY_PATH.endswith(".parquet") \
        else pd.read_csv(SUMMARY_PATH)
else:
    metrics = pd.read_parquet(f"{prefix}/metrics.parquet")

print("shape:", metrics.shape)
print("columns:", list(metrics.columns))
metrics.head()

## Coverage first

Before ranking anything, check that every configuration was scored on every
scenario. A ragged table is the most common reason a leaderboard is wrong, and it
does not announce itself.

In [ ]:
coverage = (
    metrics.groupby(["model", "top_n_features", "preprocessing_id", "hyperparam_id"])["scenario"]
    .nunique()
    .rename("scenarios_scored")
    .reset_index()
)
expected = len(cfg.scenarios)
ragged = coverage[coverage["scenarios_scored"] < expected]

print(f"{len(coverage)} configuration(s), expected {expected} scenario(s) each")
print(f"{len(ragged)} incomplete configuration(s)")
ragged.head(20)

## The leaderboard

In [ ]:
from trust_score_05.ml.evaluation import leaderboard

board = leaderboard(
    metrics,
    metric=METRIC,
    level=LEVEL,
    k_percent=K_PERCENT,
    require_all_scenarios=True,
)

print("shape:", board.shape)
board.head(25)

## The champion

`select_best_config` returns a `BestConfig`, which carries enough to reconstruct
the run: the model, the feature-list length, the preprocessing id, the
hyperparameter id and the metric value it won on. That is the object to write
into a model card.

In [ ]:
from trust_score_05.ml.evaluation import select_best_config

best = select_best_config(
    metrics,
    metric=METRIC,
    level=LEVEL,
    k_percent=K_PERCENT,
    require_all_scenarios=True,
)

print("model            :", best.model)
print("top n features   :", best.top_n_features)
print("preprocessing id :", best.preprocessing_id)
print("hyperparam id    :", best.hyperparam_id)
print("metric           :", f"{best.metric} = {best.value:.4f}")

## Is the winner actually better?

The gap between first and fifth is the number that decides whether this is a
result or a coin flip. If the top ten are within noise of each other, prefer the
simplest — fewer features, a cheaper model, a preprocessing recipe that is easier
to explain — because you will be maintaining it, not admiring it.

In [ ]:
top = board.head(10).copy()
if "value" in top.columns:
    spread = top["value"].max() - top["value"].min()
    print(f"top-10 spread: {spread:.4f}")
    print(f"relative     : {spread / max(top['value'].max(), 1e-9):.2%}")
top

## Per-scenario stability

In [ ]:
import matplotlib.pyplot as plt

from trust_score_05.common.viz.plots import configure_matplotlib

configure_matplotlib()

pivot = (
    metrics[(metrics["level"] == LEVEL) & (metrics["metric"] == METRIC)]
    .pivot_table(index="model", columns="scenario", values="value", aggfunc="max")
)

figure, axis = plt.subplots(figsize=(9, 0.45 * max(len(pivot), 4) + 2))
image = axis.imshow(pivot.values, aspect="auto")
axis.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha="right")
axis.set_yticks(range(len(pivot.index)), pivot.index)
axis.set_title(f"best {METRIC} per model per scenario ({LEVEL} level)")
figure.colorbar(image, ax=axis)
plt.tight_layout()
plt.show()

pivot

## Sensitivity to the choices

The leaderboard depends on three arbitrary decisions: the metric, the
aggregation level and k. If the champion changes when you change any of them,
that is the most important finding in this notebook and it belongs in the write-up.

In [ ]:
for level in cfg.aggregation_levels:
    for k_percent in [0.5, 1.0, 5.0]:
        try:
            candidate = select_best_config(
                metrics, metric=METRIC, level=level, k_percent=k_percent,
                require_all_scenarios=True,
            )
            print(f"level={level:10s} k={k_percent:4.1f}%  ->  "
                  f"{candidate.model:22s} top_n={candidate.top_n_features:4d} "
                  f"{candidate.preprocessing_id:16s} {candidate.value:.4f}")
        except Exception as exc:
            print(f"level={level:10s} k={k_percent:4.1f}%  ->  {type(exc).__name__}: {exc}")

## Save the figures

`save_plot` writes every format in `PLOT_FORMATS` and, when given `plot_data`,
writes the underlying numbers next to the image. That pairing is deliberate: a
plot in a deck six months from now is unfalsifiable without the data behind it.
`safe_plot_name` sanitises the filename — it had two bugs of its own, recorded in
the ML findings document, both to do with names that collided after sanitising.

In [ ]:
from trust_score_05.common.viz.plots import PLOT_FORMATS, save_plot

print("formats:", PLOT_FORMATS)

SAVE = False
if SAVE:
    written = save_plot(
        figure,
        prefix=f"{cfg.base_path}/reports/figures",
        name=f"{METRIC}-by-model-and-scenario",
        plot_data=pivot,
    )
    print("wrote:", written)
else:
    print("SAVE is False -- nothing written.")

## What next

`3.4-drift.ipynb` for the champion's drift profile. Then the model card, which is
prose, and belongs in `docs/`.